<a href="https://colab.research.google.com/github/nguyenxuandinhit/TH_Deep_Learning/blob/main/Deep_Tuan3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Flatten, Dropout
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)

In [ ]:
# ==================== BÀI TẬP 1: CIFAR-10 ====================

# 1.1 Đọc dữ liệu CIFAR-10
(X_train_c10, y_train_c10), (X_test_c10, y_test_c10) = tf.keras.datasets.cifar10.load_data()

label_names_cifar10 = ['airplane','automobile','bird','cat','deer',
                       'dog','frog','horse','ship','truck']

print('X_train shape:', X_train_c10.shape)   # (50000, 32, 32, 3)
print('X_test shape :', X_test_c10.shape)    # (10000, 32, 32, 3)
print('y_train shape:', y_train_c10.shape)

In [ ]:
# 1.2 Trực quan hóa dữ liệu
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()
for i in range(10):
    axes[i].imshow(X_train_c10[i])
    axes[i].set_title(f'Class {y_train_c10[i][0]}: {label_names_cifar10[y_train_c10[i][0]]}')
    axes[i].axis('off')
plt.suptitle('CIFAR-10 Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 1.3 Tiền xử lý dữ liệu
# Reshape: 32x32x3 → 3072 (flatten)
X_train_c10_flat = X_train_c10.reshape(50000, 32*32*3).astype('float32') / 255.0
X_test_c10_flat  = X_test_c10.reshape(10000, 32*32*3).astype('float32') / 255.0

# Flatten label (50000,1) → (50000,)
y_train_c10 = y_train_c10.flatten()
y_test_c10  = y_test_c10.flatten()

print('X_train_flat shape:', X_train_c10_flat.shape)
print('Number of classes  :', len(np.unique(y_train_c10)))

In [ ]:
# 1.4 Xây dựng mô hình ANN cho CIFAR-10
cifar10_model = Sequential([
    Dense(512, input_dim=X_train_c10_flat.shape[1],
          kernel_initializer='uniform', activation='relu'),
    Dropout(0.3),
    Dense(256, kernel_initializer='uniform', activation='relu'),
    Dropout(0.3),
    Dense(128, kernel_initializer='uniform', activation='relu'),
    Dense(10,  kernel_initializer='uniform', activation='softmax')
])

cifar10_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
cifar10_model.summary()

In [ ]:
# 1.5 Huấn luyện mô hình CIFAR-10
cifar10_fit = cifar10_model.fit(
    X_train_c10_flat, y_train_c10,
    validation_split=0.1,
    epochs=20,
    batch_size=128,
    verbose=1
)

In [ ]:
# 1.6 Đánh giá mô hình CIFAR-10
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(cifar10_fit.history['accuracy'], label='train')
ax1.plot(cifar10_fit.history['val_accuracy'], label='val')
ax1.set_title('CIFAR-10 Model Accuracy')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(cifar10_fit.history['loss'], label='train')
ax2.plot(cifar10_fit.history['val_loss'], label='val')
ax2.set_title('CIFAR-10 Model Loss')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

test_loss, test_acc = cifar10_model.evaluate(X_test_c10_flat, y_test_c10, verbose=0)
print(f'\nTest Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}')

In [ ]:
# 1.7 Dự báo ảnh mới (CIFAR-10)
preds_c10 = np.argmax(cifar10_model.predict(X_test_c10_flat[:5]), axis=1)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_test_c10[i])
    ax.set_title(f'Pred: {label_names_cifar10[preds_c10[i]]}\nTrue: {label_names_cifar10[y_test_c10[i]]}')
    ax.axis('off')
plt.suptitle('CIFAR-10 Predictions', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== BÀI TẬP 2: MNIST ====================

# 2.1 Đọc dữ liệu MNIST
(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = tf.keras.datasets.mnist.load_data()

print('X_train shape:', X_train_mnist.shape)   # (60000, 28, 28)
print('X_test shape :', X_test_mnist.shape)
print('Classes      :', np.unique(y_train_mnist))

In [ ]:
# 2.2 Trực quan hóa
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()
for i in range(10):
    axes[i].imshow(X_train_mnist[i], cmap='gray')
    axes[i].set_title(f'Label: {y_train_mnist[i]}')
    axes[i].axis('off')
plt.suptitle('MNIST Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Tiền xử lý
X_train_mnist_flat = X_train_mnist.reshape(60000, 784).astype('float32') / 255.0
X_test_mnist_flat  = X_test_mnist.reshape(10000, 784).astype('float32') / 255.0

print('X_train_flat shape:', X_train_mnist_flat.shape)

In [ ]:
# 2.4 Xây dựng mô hình ANN cho MNIST
mnist_model = Sequential([
    Dense(256, input_dim=784, kernel_initializer='uniform', activation='relu'),
    Dropout(0.2),
    Dense(128, kernel_initializer='uniform', activation='relu'),
    Dense(10,  kernel_initializer='uniform', activation='softmax')
])

mnist_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
mnist_model.summary()

In [ ]:
# 2.5 Huấn luyện MNIST model
mnist_fit = mnist_model.fit(
    X_train_mnist_flat, y_train_mnist,
    validation_split=0.1,
    epochs=20,
    batch_size=128,
    verbose=1
)

In [ ]:
# 2.6 Đánh giá MNIST model
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(mnist_fit.history['accuracy'], label='train')
ax1.plot(mnist_fit.history['val_accuracy'], label='val')
ax1.set_title('MNIST Model Accuracy'); ax1.legend()

ax2.plot(mnist_fit.history['loss'], label='train')
ax2.plot(mnist_fit.history['val_loss'], label='val')
ax2.set_title('MNIST Model Loss'); ax2.legend()
plt.tight_layout(); plt.show()

test_loss, test_acc = mnist_model.evaluate(X_test_mnist_flat, y_test_mnist, verbose=0)
print(f'Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}')

In [ ]:
# 2.7 Dự báo (MNIST)
preds_mnist = np.argmax(mnist_model.predict(X_test_mnist_flat[:10]), axis=1)
print('Predicted:', preds_mnist)
print('True     :', y_test_mnist[:10])

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_test_mnist[i], cmap='gray')
    color = 'green' if preds_mnist[i] == y_test_mnist[i] else 'red'
    ax.set_title(str(preds_mnist[i]), color=color)
    ax.axis('off')
plt.suptitle('MNIST Predictions (green=correct, red=wrong)')
plt.tight_layout(); plt.show()

In [ ]:
# ==================== BÀI TẬP 3: CAT vs DOG ====================
# Nếu chưa có dataset, tải từ Kaggle (cần API key) hoặc dùng TF datasets
# Ở đây dùng tensorflow_datasets để demo không cần file thủ công

import tensorflow_datasets as tfds

# Tải cats_vs_dogs dataset
(ds_train, ds_test), ds_info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)

IMG_SIZE = (64, 64)
BATCH_SIZE = 64

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

ds_train = ds_train.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test  = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print('Dataset loaded. Labels: 0=Cat, 1=Dog')

In [ ]:
# 3.2 Trực quan hóa Cat vs Dog
label_names_catdog = ['Cat', 'Dog']

sample_batch = next(iter(ds_train))
images_sample, labels_sample = sample_batch

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()
for i in range(10):
    axes[i].imshow(images_sample[i].numpy())
    axes[i].set_title(label_names_catdog[labels_sample[i].numpy()])
    axes[i].axis('off')
plt.suptitle('Cat vs Dog Samples')
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Xây dựng ANN Cat vs Dog
catdog_model = Sequential([
    Flatten(input_shape=(64, 64, 3)),
    Dense(512, activation='relu', kernel_initializer='uniform'),
    Dropout(0.4),
    Dense(256, activation='relu', kernel_initializer='uniform'),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_initializer='uniform'),
    Dense(1, activation='sigmoid')   # binary: 0=cat, 1=dog
])

catdog_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
catdog_model.summary()

In [ ]:
# 3.4 Huấn luyện Cat vs Dog
catdog_fit = catdog_model.fit(
    ds_train,
    validation_data=ds_test,
    epochs=15,
    verbose=1
)

In [ ]:
# 3.5 Đánh giá Cat vs Dog
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(catdog_fit.history['accuracy'], label='train')
ax1.plot(catdog_fit.history['val_accuracy'], label='val')
ax1.set_title('Cat vs Dog - Accuracy'); ax1.legend()

ax2.plot(catdog_fit.history['loss'], label='train')
ax2.plot(catdog_fit.history['val_loss'], label='val')
ax2.set_title('Cat vs Dog - Loss'); ax2.legend()
plt.tight_layout(); plt.show()

test_loss, test_acc = catdog_model.evaluate(ds_test, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')

In [ ]:
# 3.6 Dự báo Cat vs Dog
test_imgs, test_labels = next(iter(ds_test))
preds_catdog = (catdog_model.predict(test_imgs[:8]) > 0.5).astype(int).flatten()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()
for i in range(8):
    axes[i].imshow(test_imgs[i].numpy())
    true_label = label_names_catdog[test_labels[i].numpy()]
    pred_label = label_names_catdog[preds_catdog[i]]
    color = 'green' if true_label == pred_label else 'red'
    axes[i].set_title(f'Pred: {pred_label}\nTrue: {true_label}', color=color)
    axes[i].axis('off')
plt.suptitle('Cat vs Dog Predictions')
plt.tight_layout(); plt.show()

In [ ]:
# ==================== BÀI TẬP 4: ADULT INCOME ====================

# 4.1 Đọc dữ liệu Adult từ UCI (hoặc file giảng viên cung cấp)
# Nếu có file local: adult = pd.read_csv('/content/adult.csv')
# Ở đây dùng URL trực tiếp từ UCI

col_names = ['age','workclass','fnlwgt','education','education_num',
             'marital_status','occupation','relationship','race','sex',
             'capital_gain','capital_loss','hours_per_week','native_country','income']

adult = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
    names=col_names, sep=', ', engine='python', na_values='?'
)

print('Shape:', adult.shape)
adult.head()

In [ ]:
# 4.2 Khám phá dữ liệu
print('Thu nhập phân bố:')
print(adult['income'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
adult['income'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','coral'])
axes[0].set_title('Phân bố thu nhập'); axes[0].set_xlabel('Income')

adult.boxplot(column='age', by='income', ax=axes[1])
axes[1].set_title('Tuổi theo nhóm thu nhập')
plt.tight_layout(); plt.show()

In [ ]:
# 4.3 Tiền xử lý dữ liệu
adult_clean = adult.dropna().copy()

# Encode categorical columns
cat_cols = ['workclass','education','marital_status','occupation',
            'relationship','race','sex','native_country']

le = LabelEncoder()
for col in cat_cols:
    adult_clean[col] = le.fit_transform(adult_clean[col])

# Encode nhãn: <=50K → 0, >50K → 1
adult_clean['income'] = (adult_clean['income'] == '>50K').astype(int)

X_adult = adult_clean.drop('income', axis=1).values
y_adult = adult_clean['income'].values

# Feature scaling
scaler_adult = MinMaxScaler()
X_adult = scaler_adult.fit_transform(X_adult)

# Tách train/test
X_train_adult, X_test_adult, y_train_adult, y_test_adult = train_test_split(
    X_adult, y_adult, test_size=0.2, random_state=42
)

print('Train size:', X_train_adult.shape)
print('Test  size:', X_test_adult.shape)
print('Features  :', X_train_adult.shape[1])

In [ ]:
# 4.4 Xây dựng ANN
adult_model = Sequential([
    Dense(64, input_dim=X_train_adult.shape[1],
          kernel_initializer='uniform', activation='relu'),
    Dropout(0.3),
    Dense(32, kernel_initializer='uniform', activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')   # binary output
])

adult_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
adult_model.summary()

In [ ]:
# 4.5 Huấn luyện
adult_fit = adult_model.fit(
    X_train_adult, y_train_adult,
    validation_split=0.1,
    epochs=30,
    batch_size=256,
    verbose=1
)

In [ ]:
# 4.6 Đánh giá Adult Model
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(adult_fit.history['accuracy'], label='train')
ax1.plot(adult_fit.history['val_accuracy'], label='val')
ax1.set_title('Adult Income - Accuracy'); ax1.legend()

ax2.plot(adult_fit.history['loss'], label='train')
ax2.plot(adult_fit.history['val_loss'], label='val')
ax2.set_title('Adult Income - Loss'); ax2.legend()
plt.tight_layout(); plt.show()

y_pred_adult = (adult_model.predict(X_test_adult) > 0.5).astype(int).flatten()
print('Test Accuracy:', accuracy_score(y_test_adult, y_pred_adult))
print('\nClassification Report:')
print(classification_report(y_test_adult, y_pred_adult,
                            target_names=['<=50K', '>50K']))

In [ ]:
# 4.7 Dự báo người mới
# Ví dụ: 1 mẫu từ tập test
sample_idx = 0
sample_input = X_test_adult[sample_idx].reshape(1, -1)
prob = adult_model.predict(sample_input)[0][0]
pred_class = '>50K' if prob > 0.5 else '<=50K'
true_class = '>50K' if y_test_adult[sample_idx] == 1 else '<=50K'
print(f'Xác suất >50K: {prob:.4f}')
print(f'Dự báo: {pred_class} | Thực tế: {true_class}')

In [ ]:
# ==================== BÀI TẬP 5: CAR EVALUATION ====================

# 5.1 Đọc dữ liệu Car (UCI)
car_cols = ['buying','maint','doors','persons','lug_boot','safety','class']

car = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data',
    names=car_cols
)

print('Shape:', car.shape)
print('Classes:', car['class'].unique())
car.head()

In [ ]:
# 5.2 Khám phá dữ liệu
print('Phân bố nhãn:')
print(car['class'].value_counts())

car['class'].value_counts().plot(kind='bar', color=['#e74c3c','#3498db','#2ecc71','#f39c12'])
plt.title('Phân bố chất lượng xe')
plt.xlabel('Class'); plt.ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# 5.3 Tiền xử lý Car Dataset
car_clean = car.copy()

# Encode tất cả cột (đều là categorical)
le_car = LabelEncoder()
for col in car_clean.columns:
    car_clean[col] = le_car.fit_transform(car_clean[col])

# Lưu label encoder riêng cho cột class
le_class = LabelEncoder()
car['class_enc'] = le_class.fit_transform(car['class'])

X_car = car_clean.drop('class', axis=1).values
y_car = car_clean['class'].values

# One-hot encode cho y (vì 4 lớp)
y_car_ohe = tf.keras.utils.to_categorical(y_car, num_classes=4)

X_train_car, X_test_car, y_train_car, y_test_car = train_test_split(
    X_car, y_car_ohe, test_size=0.2, random_state=42
)

print('Train size:', X_train_car.shape)
print('Test  size:', X_test_car.shape)
print('Num classes:', y_car_ohe.shape[1])

In [ ]:
# 5.4 Xây dựng mô hình ANN
car_model = Sequential([
    Dense(64, input_dim=X_train_car.shape[1],
          kernel_initializer='uniform', activation='relu'),
    Dropout(0.2),
    Dense(32, kernel_initializer='uniform', activation='relu'),
    Dense(4, activation='softmax')   # 4 classes
])

car_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
car_model.summary()

In [ ]:
# 5.5 Huấn luyện
car_fit = car_model.fit(
    X_train_car, y_train_car,
    validation_split=0.1,
    epochs=50,
    batch_size=32,
    verbose=1
)

In [ ]:
# 5.6 Đánh giá Car Model
class_names_car = ['acc', 'good', 'unacc', 'vgood']  # thứ tự sau LabelEncoder

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(car_fit.history['accuracy'], label='train')
ax1.plot(car_fit.history['val_accuracy'], label='val')
ax1.set_title('Car Quality - Accuracy'); ax1.legend()

ax2.plot(car_fit.history['loss'], label='train')
ax2.plot(car_fit.history['val_loss'], label='val')
ax2.set_title('Car Quality - Loss'); ax2.legend()
plt.tight_layout(); plt.show()

y_pred_car = np.argmax(car_model.predict(X_test_car), axis=1)
y_true_car = np.argmax(y_test_car, axis=1)
print('Test Accuracy:', accuracy_score(y_true_car, y_pred_car))
print('\nClassification Report:')
print(classification_report(y_true_car, y_pred_car, target_names=class_names_car))

In [ ]:
# 5.7 Dự báo xe mới
# Ví dụ từ tập test
sample_car = X_test_car[0].reshape(1, -1)
prob_car = car_model.predict(sample_car)[0]
pred_car = class_names_car[np.argmax(prob_car)]
true_car = class_names_car[y_true_car[0]]

print('Xác suất theo từng lớp:')
for i, cls in enumerate(class_names_car):
    print(f'  {cls}: {prob_car[i]:.4f}')
print(f'\nDự báo: {pred_car} | Thực tế: {true_car}')